# IMPORTS

In [1]:
import time
time_start = time.time()
import glob
import os
import shutil
import h5py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import random

from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn.functional as F
import albumentations as A

# SETTINGS

In [2]:
# Session Settings

SESSION_ID = "model04_kaggle_submission_10"
DATASET_PATH = "/home/dinis/Desktop/Universidade/Cadeiras/Ano4-Q1/AE4353 Artificial Intelligence for Aerospace Control and Operations/Competition/Data/ae4353-2026"
OUTPUT_BASE_PATH = "/home/dinis/Desktop/Universidade/Cadeiras/Ano4-Q1/AE4353 Artificial Intelligence for Aerospace Control and Operations/Competition/Output"
SEED = 2006

# Dataset Settings

NUM_WORKERS = 4
BATCH_SIZE = 8
VAL_SPLIT = 0.2

# Criterion Settings

WEIGHT_DICE = 0.001

# Model Settings

INPUT_CHANNELS = 3 # Ash Colors
OUTPUT_CHANNELS = 1 
CONV_KERNEL_SIZE = 3
POOL_KERNEL_SIZE = 2
POOL_STRIDE = 2
UPSAMPLE_SCALE = 1.0

# Optimizer & Scheduler Settings

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2
MIN_LEARNING_RATE = 1e-6

# Training Settings

MAX_EPOCHS = 1

# Validation Settings

THRESHOLD_RANGE = np.arange(0.1, 0.9, 0.05)


# INITIALIZATION

In [3]:
# Import all necessaryfiles from the dataset folder
if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(f"There is no folder at {DATASET_PATH}.")

shutil.copy(f"{DATASET_PATH}/metric.py", "metric.py")
from metric import rle_encode, rle_decode, global_dice, write_submission

train_files = sorted(glob.glob(f"{DATASET_PATH}/train_*.h5"))
test_files = sorted(glob.glob(f"{DATASET_PATH}/test_*.h5"))

# Create output folder for this session
OUTPUT_PATH = os.path.join(OUTPUT_BASE_PATH, SESSION_ID)
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Seed Everything
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)                     # seeds the GPU as well
torch.backends.cudnn.benchmark = False      # the same algorithms on every run
torch.backends.cudnn.deterministic = True

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator().manual_seed(SEED)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOAD DATASETS

In [4]:
class ContrailDataset(Dataset):
    
    def __init__(self, file_paths, is_test=False, do_augmentation=False, seed=SEED):
        
        self.file_paths = sorted(file_paths)

        self.is_test = is_test

        self.do_augmentation = do_augmentation

        # Using an index map to save memory and avoid unnecessary loading of all data at once
        self.index_map = []

        # Dictionary to hold open HDF5 file handles to avoid reopening files multiple times
        self.handles = {}
        
        # Cache (scale, offset) per file to speed up __getitem__
        self.file_attrs = {}  

        if self.do_augmentation:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Affine(scale=(0.9, 1.1), translate_percent=(-0.0625, 0.0625), rotate=(-45, 45), p=0.5),
            ], seed=seed)

        for path in self.file_paths:
            with h5py.File(path, 'r') as f:
                num_samples = f['bands'].shape[0]
                scale = float(f['bands'].attrs['scale'])
                offset = float(f['bands'].attrs['offset'])
                self.file_attrs[path] = (scale, offset)
                self.index_map.extend([(path, idx) for idx in range(num_samples)])

    def __len__(self):
        return len(self.index_map)

    def _get_handle(self, path):
        # Avoid reopening the same HDF5 file multiple times in a multi-worker setting
        if path not in self.handles:
            self.handles[path] = h5py.File(path, 'r')
        return self.handles[path]

    @staticmethod
    def normalize_range(data, f_min, f_max):
        '''
        Normalize the data to the range [0, 1] based on the provided min and max values.
        '''
        return np.clip((data - f_min) / (f_max - f_min), 0.0, 1.0)

    def convert_to_ash_false_color(self, raw_bands, scale, offset):
        '''
        Convert the raw bands to Ash False Color.

        STANDARD ASH FALSE-COLOR RECIPE (Ash RGB) -> Substitutions for the available bands in the dataset:
        
        RED: 12.3 µm - 10.3 µm -> Band 15 (12.3 µm) - Band 14 (11.2 µm)
          Range [Min, Max]: [-4 K, 2 K], Gamma = 1.0
        
        GREEN: 11.2 µm - 8.4 µm -> Band 14 (11.2 µm) - Band 11 (8.4 µm)
          Range [Min, Max]: [-4 K, 5 K], Gamma = 1.0
        
        BLUE: 10.3 µm -> Band 14 (11.2 µm)
          Range [Min, Max]: [243 K, 303 K], Gamma = 1.0
        '''
        
        kelvin = raw_bands.astype(np.float32) * scale + offset

        red = self.normalize_range(kelvin[2] - kelvin[1], -4.0, 2.0)
        green = self.normalize_range(kelvin[1] - kelvin[0], -4.0, 5.0)
        blue = self.normalize_range(kelvin[1], 243.0, 303.0)

        # Tensor with shape (H, W, C = 3)
        return np.stack([red, green, blue], axis=-1).astype(np.float32)

    def __getitem__(self, idx):

        # Load the sample
        path, local_idx = self.index_map[idx]
        data = self._get_handle(path)
        raw_bands = data['bands'][local_idx]
        scale, offset = self.file_attrs[path]

        # Convert to Ash False Color
        ash_image = self.convert_to_ash_false_color(raw_bands, scale, offset)

        if self.is_test:
            # Return the image tensor and its corresponding ID for test data

            # Convert to PyTorch tensor with shape (C, H, W)
            image_tensor = torch.from_numpy(np.ascontiguousarray(ash_image.transpose(2, 0, 1)))

            image_id = data['id'][local_idx].decode('utf-8')
            return image_tensor, image_id

        else:
            # Return the image tensor and its corresponding mask for training/validation data

            mask = data['mask'][local_idx].astype(np.float32)

            if self.do_augmentation:
                augmented = self.transform(image=ash_image, mask=mask)
                ash_image = augmented['image']
                mask = augmented['mask']

            # Convert to PyTorch tensor with shape (C, H, W)
            image_tensor = torch.from_numpy(np.ascontiguousarray(ash_image.transpose(2, 0, 1)))
            mask_tensor = torch.from_numpy(np.ascontiguousarray(mask)).unsqueeze(0)

            return image_tensor, mask_tensor

In [5]:
# Calculate the total number of training images
total_train_images = sum([len(h5py.File(f, 'r')['bands']) for f in train_files])

# Randomly shuffle the indices of the training images
total_train_indices = torch.randperm(total_train_images, generator=g).tolist()

val_size = int(total_train_images * VAL_SPLIT)
train_size = total_train_images - val_size
val_indices = total_train_indices[:val_size]
train_indices = total_train_indices[val_size:]

# Train Loader
total_train_dataset = ContrailDataset(train_files, is_test=False, do_augmentation=True, seed=SEED)
train_dataset = Subset(total_train_dataset, train_indices)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, worker_init_fn=seed_worker, generator=g, persistent_workers=True, pin_memory=True)

# Validation Loader
total_val_dataset = ContrailDataset(train_files, is_test=False, do_augmentation=False, seed=SEED)
val_dataset = Subset(total_val_dataset, val_indices)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, worker_init_fn=seed_worker, generator=g, persistent_workers=True, pin_memory=True)

# Test Loader
test_dataset = ContrailDataset(test_files, is_test=True, do_augmentation=False, seed=SEED)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, worker_init_fn=seed_worker, generator=g, persistent_workers=True, pin_memory=True)

# Pin memory for faster data transfer to GPU
# Persistent workers to keep the data loading workers alive across epochs and avoid the loading of the files

# CRITERION

In [6]:
class BCELoss(nn.Module):
    
    def __init__(self, is_logits=True):
        super().__init__()
        self.bce_loss = nn.BCEWithLogitsLoss() if is_logits else nn.BCELoss()

    def forward(self, preds, targets):
        return self.bce_loss(preds, targets)

class DiceLoss(nn.Module):

    def __init__(self, is_logits=True, tol=1e-6):
        super().__init__()
        self.is_logits = is_logits
        self.tol = tol

    def forward(self, preds, targets):

        if self.is_logits:
            # Unbounded -> bounded [0, 1]
            preds = torch.sigmoid(preds)
        
        # Flatten to (B*C*H*W) for each image in the batch
        preds = preds.reshape(-1)
        targets = targets.reshape(-1)

        intersection = (preds * targets).sum()
        cardinality = preds.sum() + targets.sum()
        dice_score = (2. * intersection + self.tol) / (cardinality + self.tol) # Avoid division by zero

        return 1. - dice_score


class DiceBCELoss(nn.Module):
    def __init__(self, is_logits=True, tol=1e-6, weight_dice=0.5):
        super().__init__()
        self.weight_dice = weight_dice
        self.dice_loss = DiceLoss(is_logits=is_logits, tol=tol)
        self.bce_loss = BCELoss(is_logits=is_logits)

    def forward(self, preds, targets):
        dice_loss = self.dice_loss(preds, targets)
        bce_loss = self.bce_loss(preds, targets)
        weighted_loss = self.weight_dice * dice_loss + bce_loss

        return weighted_loss, dice_loss, bce_loss

In [7]:
criterion = DiceBCELoss(is_logits=True, tol=1e-6, weight_dice=WEIGHT_DICE)

# MODEL

In [8]:
class Contrail_CNN(nn.Module):
    
    """
    U-net style architecture.
      - Encoder: 4 sequential blocks of Conv2d -> BatchNorm2d -> ReLU,
        each followed by 2x2 MaxPool2d downsampling (channels: input -> 64 -> 128 -> 256 -> 512).
      - Bottleneck: Conv2d -> BatchNorm2d -> ReLU maintaining the 16x16 feature map at 512 channels.
      - Decoder: 4 sequential stages using ConvTranspose2d (2x stride) paired with channel concatenation of encoder skip-connections.
      - Output Head: 1x1 Conv2d mapping the final 64 feature maps to `output_channels` (raw logits).

    Shape (default 256x256 input):
      - Input:  (Batch, input_channels, 256, 256)
      - Output: (Batch, output_channels, 256, 256)
    """

    def __init__(self, input_channels=3, output_channels=1, conv_kernel_size=3, pool_kernel_size=2, pool_stride=2, upsample_scale=1.0):
        
        super().__init__()
        self.upsample_scale = upsample_scale

        # --- Encoder ---

        self.enc1 = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=conv_kernel_size, padding="same"),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.pool1 = nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride)

        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=conv_kernel_size, padding="same"),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.pool2 = nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride)

        self.enc3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=conv_kernel_size, padding="same"),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.pool3 = nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride)

        self.enc4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=conv_kernel_size, padding="same"),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        self.pool4 = nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride)

        # --- Bottleneck ---

        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=conv_kernel_size, padding="same"),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )

        # --- Decoder ---
        # All Conv2d have input channels equal to the sum of the upsampled feature maps and the corresponding encoder skip-connection feature maps.

        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=pool_kernel_size, stride=pool_stride)
        self.dec4 = nn.Sequential(
            nn.Conv2d(256 + 512, 256, kernel_size=conv_kernel_size, padding="same"),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=pool_kernel_size, stride=pool_stride)
        self.dec3 = nn.Sequential(
            nn.Conv2d(128 + 256, 128, kernel_size=conv_kernel_size, padding="same"),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=pool_kernel_size, stride=pool_stride)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64 + 128, 64, kernel_size=conv_kernel_size, padding="same"),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=pool_kernel_size, stride=pool_stride)
        self.dec1 = nn.Sequential(
            nn.Conv2d(64 + 64, 64, kernel_size=conv_kernel_size, padding="same"),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.out_conv = nn.Conv2d(64, output_channels, kernel_size=1)

    def forward(self, x):

        # Extract HxW from tensor shape (B, C, H, W)
        original_size = x.shape[-2:]  # (256, 256)

        # Upsample
        if self.upsample_scale != 1.0:
            x = F.interpolate(x, scale_factor=self.upsample_scale, mode="bilinear", align_corners=False)

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        # Bottleneck
        b = self.bottleneck(self.pool4(e4))

        # Decoder
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        # Output
        logits = self.out_conv(d1)

        # Downsample
        if self.upsample_scale != 1.0:
            logits = F.interpolate(logits, size=original_size, mode="bilinear", align_corners=False)

        return logits

In [9]:
model = Contrail_CNN(input_channels=INPUT_CHANNELS, output_channels=OUTPUT_CHANNELS, conv_kernel_size=CONV_KERNEL_SIZE, pool_kernel_size=POOL_KERNEL_SIZE, pool_stride=POOL_STRIDE, upsample_scale=UPSAMPLE_SCALE)

model = model.to(device)

# OPTIMIZER & SCHEDULER

In [10]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE)

# TRAINING

In [11]:
# Pre-allocate threshold tensor on GPU: shape (1, size, 1, 1)
thresholds_gpu = torch.tensor(THRESHOLD_RANGE, device=device, dtype=torch.float32).view(1, -1, 1, 1)
num_thresholds = len(THRESHOLD_RANGE)

# All metrics
weighted_loss_history = []
dice_loss_history = []
bce_loss_history = []
epoch_full_history = []
learning_rate_history = []

validation_history = {
    "best_dice": [],
    "best_threshold": [],
    "false_positives": [],
    "false_negatives": [],
    "precision": [],
    "recall": []
}

best_epoch = 1
best_global_dice = 0.0
best_threshold = 0.0

for epoch in range(MAX_EPOCHS):
    
    # --- 1. Training ---

    model.train()
    running_loss = torch.zeros(1, device=device)
    running_dice_loss = torch.zeros(1, device=device)
    running_bce_loss = torch.zeros(1, device=device)

    train_loop = tqdm(train_loader, desc=f"Epoch [{epoch + 1}/{MAX_EPOCHS}] Training", leave=False)

    for images, masks in train_loop:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        weighted_loss, dice_loss, bce_loss = criterion(outputs, masks)
        weighted_loss.backward()
        optimizer.step()

        # GPU 
        running_loss += weighted_loss.detach()
        running_dice_loss += dice_loss.detach()
        running_bce_loss += bce_loss.detach()

    # GPU-CPU sync
    num_batches = len(train_loader)
    avg_loss = (running_loss / num_batches).item()
    avg_dice_loss = (running_dice_loss / num_batches).item()
    avg_bce_loss = (running_bce_loss / num_batches).item()

    weighted_loss_history.append(avg_loss)
    dice_loss_history.append(avg_dice_loss)
    bce_loss_history.append(avg_bce_loss)

    # --- 2. Validation ---

    model.eval()
    tp_counts = torch.zeros(num_thresholds, dtype=torch.long, device=device)
    fp_counts = torch.zeros(num_thresholds, dtype=torch.long, device=device)
    fn_counts = torch.zeros(num_thresholds, dtype=torch.long, device=device)

    val_loop = tqdm(val_loader, desc=f"Epoch [{epoch + 1}/{MAX_EPOCHS}] Validation", leave=False)

    with torch.no_grad():
        for images, masks in val_loop:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            # 1. Original
            probs_original = torch.sigmoid(model(images))

            # 2. Flips (Horizontal & Vertical)
            probs_hflip = torch.flip(torch.sigmoid(model(torch.flip(images, dims=[3]))), dims=[3])
            probs_vflip = torch.flip(torch.sigmoid(model(torch.flip(images, dims=[2]))), dims=[2])

            # 3. Rotations (90°, 180°, 270°) — rotate input by +k, un-rotate output by -k
            probs_rot90  = torch.rot90(torch.sigmoid(model(torch.rot90(images, k=1, dims=[2, 3]))), k=-1, dims=[2, 3])
            probs_rot180 = torch.rot90(torch.sigmoid(model(torch.rot90(images, k=2, dims=[2, 3]))), k=-2, dims=[2, 3])
            probs_rot270 = torch.rot90(torch.sigmoid(model(torch.rot90(images, k=3, dims=[2, 3]))), k=-3, dims=[2, 3])

            # Average all 6 predictions
            probs = (probs_original + probs_hflip + probs_vflip +probs_rot90 + probs_rot180 + probs_rot270) / 6.0

            preds_bool = (probs > thresholds_gpu)
            targets_bool = masks.bool()

            # GPU, sum over batch and spatial dimensions (0, 2, 3) to get counts per threshold for all thresholds in THRESHOLD_RANGE
            tp_counts += (preds_bool & targets_bool).sum(dim=(0, 2, 3))
            fp_counts += (preds_bool & ~targets_bool).sum(dim=(0, 2, 3))
            fn_counts += (~preds_bool & targets_bool).sum(dim=(0, 2, 3))

    # Transfer the all threshold arrays to CPU once
    tp_arr = tp_counts.cpu().numpy()
    fp_arr = fp_counts.cpu().numpy()
    fn_arr = fn_counts.cpu().numpy()

    best_val_dice = 0.0
    best_val_threshold = 0.0
    best_val_false_positives = 0
    best_val_false_negatives = 0
    best_val_precision = 0.0
    best_val_recall = 0.0
    current_lr = optimizer.param_groups[0]['lr']

    for idx, t in enumerate(THRESHOLD_RANGE):
        tp = int(tp_arr[idx])
        fp = int(fp_arr[idx])
        fn = int(fn_arr[idx])

        # Equivalent to global_dice function
        denom = 2 * tp + fp + fn
        dice_score = (2.0 * tp / denom) if denom > 0 else 1.0
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        epoch_full_history.append({
            "epoch": epoch + 1,
            "avg_train_loss": avg_loss,
            "avg_loss_dice": avg_dice_loss,
            "avg_loss_bce": avg_bce_loss,
            "threshold": float(t),
            "dice": float(dice_score),
            "false_positives": fp,
            "false_negatives": fn,
            "precision": precision,
            "recall": recall,
            "learning_rate": current_lr,
        })

        if dice_score > best_val_dice:
            best_val_dice = dice_score
            best_val_threshold = float(t)
            best_val_false_positives = fp
            best_val_false_negatives = fn
            best_val_precision = precision
            best_val_recall = recall

    validation_history["best_dice"].append(best_val_dice)
    validation_history["best_threshold"].append(best_val_threshold)
    validation_history["false_positives"].append(best_val_false_positives)
    validation_history["false_negatives"].append(best_val_false_negatives)
    validation_history["precision"].append(best_val_precision)
    validation_history["recall"].append(best_val_recall)

    print(
        f"Epoch [{epoch + 1}/{MAX_EPOCHS}], Loss: {avg_loss:.4f}, "
        f"Validation Dice: {best_val_dice:.4f}, Best Threshold: {best_val_threshold:.2f}, "
        f"False Positives: {best_val_false_positives}, False Negatives: {best_val_false_negatives}, "
        f"Precision: {best_val_precision:.2f}, Recall: {best_val_recall:.2f}, "
        f"Dice Loss: {avg_dice_loss:.4f}, BCE Loss: {avg_bce_loss:.4f}, Learning Rate: {current_lr:.6f}"
    )

    if best_val_dice > best_global_dice:
        best_epoch = epoch + 1
        best_global_dice = best_val_dice
        best_threshold = best_val_threshold
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_dice': best_val_dice,
            'best_threshold': best_val_threshold,
            'torch_rng_state': torch.get_rng_state(),
            'np_rng_state': np.random.get_state(),
            'python_rng_state': random.getstate(),
            'generator_state': g.get_state()
        }
        torch.save(checkpoint, os.path.join(OUTPUT_PATH, "best_checkpoint.pth"))
        print(f"--- New best model saved with Validation Dice: {best_global_dice:.4f} ---")

    learning_rate_history.append(current_lr)
    scheduler.step()

# Save final epoch checkpoint
last_checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_dice': best_val_dice,
    'best_threshold': best_val_threshold,
    'torch_rng_state': torch.get_rng_state(),
    'np_rng_state': np.random.get_state(),
    'python_rng_state': random.getstate(),
    'generator_state': g.get_state()
}
torch.save(last_checkpoint, os.path.join(OUTPUT_PATH, "last_checkpoint.pth"))

Epoch [1/1], Loss: 0.0736, Validation Dice: 0.1585, Best Threshold: 0.10, False Positives: 113210, False Negatives: 363117, Precision: 0.28, Recall: 0.11, Dice Loss: 0.9820, BCE Loss: 0.0727, Learning Rate: 0.001000
--- New best model saved with Validation Dice: 0.1585 ---


# TEST

In [15]:
# Save metrics
metrics_df = pd.DataFrame(epoch_full_history)
csv_path = os.path.join(OUTPUT_PATH, "training_validation_metrics.csv")
metrics_df.to_csv(csv_path, index=False)
print(f"Saved metrics to {csv_path}")

# Load best model checkpoint and generate submission
best_ckpt = torch.load(os.path.join(OUTPUT_PATH, "best_checkpoint.pth"), map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
optimal_threshold = best_ckpt['best_threshold']
print(f"Loaded best checkpoint from Epoch {best_ckpt['epoch'] + 1} (Dice: {best_ckpt['best_dice']:.4f}, Threshold: {optimal_threshold:.2f})")

model.eval()
test_ids = []
test_preds = []

test_loop = tqdm(test_loader, desc="Generating Test Predictions", leave=False)

with torch.no_grad():
    for images, img_ids in test_loop:
        
        images = images.to(device, non_blocking=True)
        
        # 1. Original
        probs_original = torch.sigmoid(model(images))

        # 2. Flips (Horizontal & Vertical)
        probs_hflip = torch.flip(torch.sigmoid(model(torch.flip(images, dims=[3]))), dims=[3])
        probs_vflip = torch.flip(torch.sigmoid(model(torch.flip(images, dims=[2]))), dims=[2])

        # 3. Rotations (90°, 180°, 270°) — rotate input by +k, un-rotate output by -k
        probs_rot90  = torch.rot90(torch.sigmoid(model(torch.rot90(images, k=1, dims=[2, 3]))), k=-1, dims=[2, 3])
        probs_rot180 = torch.rot90(torch.sigmoid(model(torch.rot90(images, k=2, dims=[2, 3]))), k=-2, dims=[2, 3])
        probs_rot270 = torch.rot90(torch.sigmoid(model(torch.rot90(images, k=3, dims=[2, 3]))), k=-3, dims=[2, 3])

        # Average all 6 predictions
        probs = (probs_original + probs_hflip + probs_vflip +probs_rot90 + probs_rot180 + probs_rot270) / 6.0

        #Remove the channel dimension so masks are shape (256, 256)
        preds = (probs > optimal_threshold).cpu().numpy()
        
        test_preds.append(preds)
        test_ids.extend(img_ids)

test_preds = np.concatenate(test_preds, axis=0)
submission_path = os.path.join(OUTPUT_BASE_PATH, "submission.csv")
write_submission(record_ids=test_ids, masks=test_preds, path=submission_path)
print(f"Saved submission to {submission_path}")   

time_end = time.time()
time_taken = time_end - time_start
print(f"Notebook ran in {time_taken / 3600:.2f} hours and {(time_taken % 3600) / 60:.2f} minutes.")

Saved metrics to /home/dinis/Desktop/Universidade/Cadeiras/Ano4-Q1/AE4353 Artificial Intelligence for Aerospace Control and Operations/Competition/Output/model04_kaggle_submission_10/training_validation_metrics.csv
Loaded best checkpoint from Epoch 1 (Dice: 0.1585, Threshold: 0.10)


Saved submission to /home/dinis/Desktop/Universidade/Cadeiras/Ano4-Q1/AE4353 Artificial Intelligence for Aerospace Control and Operations/Competition/Output/submission.csv
Notebook ran in 0.22 hours and 13.11 minutes.
